<div align="center">
  <img src="https://raw.githubusercontent.com/WSU-AI-in-ME/ai-in-me-1/main/img/wsu_logo_horizontal.png" alt="Wayne State University Logo" width="320">
  <h1>Practice 8: PCA, Clustering, and Anomaly Detection Experiments</h1>
</div>

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/WSU-AI-in-ME/ai-in-me-1/blob/main/practice/week08/practice08_pca_clustering_and_anomaly_detection.ipynb)

[Course Repository](https://github.com/WSU-AI-in-ME/ai-in-me-1)

[Practice Index](https://github.com/WSU-AI-in-ME/ai-in-me-1/blob/main/Practice_index.ipynb)

### ME 5995: AI in Mechanical Engineering I: Fundamentals of Manufacturing Data Science

Wayne State University

**Practice:** Optional, ungraded, no submission

**Version:** Student Starter

Open in Colab and save a personal copy before editing.
Try each experiment before opening the [self-check solution](practice08_pca_clustering_and_anomaly_detection_solution.ipynb).

## Student Information

Optional for your personal copy. Double-click this Markdown cell to edit.

**Student name:** TODO: Enter your name  
**WSU AccessID:** TODO: Enter your WSU AccessID

## How to use this Practice

The suggested basic route has **10 activities**. Run setup, then A1 through C3
in order. Later activities reuse earlier variables. Four additional experiments
follow a clear stop point. They are optional and are not prerequisites for Lab 7.
Lab 7 will supply its own necessary instructions.

| Part | Activities | Main question |
|---|---|---|
| PCA | [A1](#a1), [A2](#a2), [A3](#a3) | What information does a low-dimensional representation preserve? |
| K-means | [B1](#b1), [B2](#b2), [B3](#b3), [B4](#b4) | How do distance, k and reference classes affect our interpretation? |
| Anomaly detection | [C1](#c1), [C2](#c2), [C3](#c3) | How can a learned normal pattern detect some anomalies and miss others? |
| Additional experiments | [D1](#d1), [D2](#d2), [D3](#d3), [D4](#d4) | When do the methods fail or depend on settings? |

Each activity contains an explanation, a completed example, a small experiment
and a short personal note. These notes are for self-study, not submission.
Restart the runtime and Run all to reproduce a fresh run. Make threshold choices
in C2 before inspecting the final test in C3.

**Learning objectives:** distinguish representation, clustering and detection;
interpret PCA scores/loadings/variance; explore K-means centroids and k; and
calibrate a normal-only detector using validation evidence.

**Software:** NumPy, pandas, Matplotlib, scikit-learn and Jupyter notebook support.
Colab supplies these packages. For local work, install the required packages
in your local Python environment. If a requirements.txt file is provided,
you may use it to install those packages. No external data files are required.

**Data and engineering context:** Every numerical dataset below is synthetic
and dimensionless. Seeds and recipes are fixed for teaching. Coordinates could
illustrate correlated measurements, operating groups or departures from normal
patterns, but they are not measured manufacturing signals or validated fault
mechanisms. Generating labels are for interpretation, not model fitting.
Small clean examples reveal mechanisms; they do not establish industrial performance.

**Keep the workload small:** change only the named setting in each activity. All data-generation, distance-update, result-table and drawing code is supplied; run it unchanged. You do not need to write or memorize that infrastructure. Start by predicting the change, run the two settings, and record one short observation. The ten basic activities can be spread across study sessions. The additional experiments are a separate optional bank. Running unchanged reproduces the initial example; make the stated comparison to practice the concept.

In [ ]:
# Run this cell first. Imports give short names to the libraries used below.
# NumPy stores feature arrays, pandas displays tables, and Matplotlib draws figures.

# PROVIDED: libraries for arrays, tables, plots and explicit learning steps.
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display
from sklearn.datasets import make_blobs, make_moons
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.cluster import KMeans
from sklearn.ensemble import IsolationForest
from sklearn.metrics import balanced_accuracy_score, confusion_matrix
from sklearn.metrics import silhouette_score, precision_score, recall_score

# Use the same readable plot defaults throughout the notebook.
plt.rcParams['axes.titlesize'] = 10
plt.rcParams['axes.labelsize'] = 10

### Read the model steps; run supplied drawings unchanged

Drawing cells are supplied after the relevant calculations, so you can focus
on the model and experimental setting first. They use explicit Matplotlib
statements rather than custom plotting functions. A color can represent a
cluster or a reference class; matching color numbers across separate models
do not establish matching groups. Run the drawing cells unchanged.

### Provided synthetic data: run unchanged

`P` contains 240 correlated observations and two features. `R` contains 240
observations of three features driven mostly by two latent directions.
`K` contains 240 points around three compact centers; `K_reference` stores their
generating group labels. No scaler, PCA or clustering model uses those labels.
`column_stack` combines feature columns and `@` applies the supplied linear
mixing matrix. These generation details are provided infrastructure.

**Reading the model code:** `fit` learns from the supplied observations;
`transform` applies the learned representation without relearning it.
`n_components` sets the number of retained PCA directions. `svd_solver='full'`
fixes the numerical PCA method for reproducibility, and `whiten=False` leaves
score variances unwhitened. A trailing underscore marks a learned attribute:
`components_` contains direction coefficients and `explained_variance_ratio_`
contains variance fractions. For K-means, `n_init=10` tries ten initializations,
`random_state` fixes the random choices, and `inertia_` reports within-cluster
squared distances. Index `[:, 0]` selects every row of feature column 0.

In [ ]:
# P and R have one observation per row; no coordinate has a physical sensor unit.
# The generators below are supplied so you can focus on PCA and clustering.

# PROVIDED: independent seeds make each teaching dataset reproducible.
rng_p = np.random.default_rng(801)
p_axis = rng_p.normal(size=240)
# Generate noise separately so the two correlated feature columns are visible.
p_noise = rng_p.normal(0, 0.55, 240)
p_feature_2 = 0.8 * p_axis + p_noise
P = np.column_stack([p_axis, p_feature_2])
rng_r = np.random.default_rng(802)
r_latent = rng_r.normal(size=(240, 2))
R = r_latent @ np.array([[1, 1, 0.25], [0.55, -0.55, 1.2]])
R = R + rng_r.normal(0, 0.04, (240, 3))
K, K_reference = make_blobs(n_samples=240, centers=[[-3, 0], [3, -2], [3, 2]],
                            cluster_std=0.45, random_state=803)
print('P shape:', P.shape, 'R shape:', R.shape, 'K shape:', K.shape)

<a id="a1"></a>
## A1: Scaling and principal directions

PCA learns directions that maximize variance. A **loading** is a direction
coefficient; a **score** is an observation's coordinate along that direction.
Neither is a causal explanation. Changing a feature's numerical scale can change
raw PCA. StandardScaler subtracts each training mean and divides by its standard
deviation. This changes the metric and is a modeling choice, not a universal rule.
Direction signs can flip without changing the subspace.

**Completed example:** Run and inspect the output.

In [ ]:
# P has shape (240, 2): 240 observations and two dimensionless features.
# Learn one direction from raw coordinates before changing their numerical scale.

# Construct PCA first; n_components controls compression and full fixes the solver.
raw_pca = PCA(n_components=1, svd_solver='full')

# Learn principal directions and their variances from P; no reference labels are used.
raw_pca.fit(P)

# raw_scores has one score column per retained PC; apply the fitted directions without relearning them.
raw_scores = raw_pca.transform(P)
print('Raw loading:', raw_pca.components_[0])
print('First three scores:', raw_scores[:3, 0])
print('Raw retained variance:', raw_pca.explained_variance_ratio_[0])

**Your experiment:** Set conversion_factor to 20. Compare the raw direction coefficients, then check whether standardization removes this pure numerical rescaling.

In [ ]:
# Compare the same observations under two coordinate scales.
# Fit each scaler separately, then compare its standardized output.

# TODO: change 1 to 20; this changes a coordinate scale, not the observations.
conversion_factor = 1
# Copy the observations so the original P remains unchanged.
P_converted = P.copy()
# [:, 0] selects feature 1 for all 240 rows; feature 2 stays unchanged.
P_converted[:, 0] = P[:, 0] * conversion_factor

# Construct PCA first; n_components controls compression and full fixes the solver.
converted_pca = PCA(n_components=1, svd_solver='full')

# Learn principal directions and their variances from P_converted; no reference labels are used.
converted_pca.fit(P_converted)

**Step 2 of 3: Standardize both coordinate systems**

In [ ]:
# Construct a scaler; fit below will learn a separate mean and scale for each feature.
p_scaler = StandardScaler()

# Learn a mean and standard deviation for each column of P; no row is transformed yet.
p_scaler.fit(P)

# P_scaled keeps the same rows and columns; subtract the fitted mean and divide by the fitted scale.
P_scaled = p_scaler.transform(P)

converted_scaler = StandardScaler()

# Learn a mean and standard deviation for each column of P_converted; no row is transformed yet.
converted_scaler.fit(P_converted)

# P_converted_scaled keeps the same rows and columns; subtract the fitted mean and divide by the fitted scale.
P_converted_scaled = converted_scaler.transform(P_converted)

scaled_pca = PCA(n_components=1, svd_solver='full')

# Learn principal directions and their variances from P_converted_scaled; no reference labels are used.
scaled_pca.fit(P_converted_scaled)
print('Converted raw loading:', converted_pca.components_[0])
print('Converted raw retained variance:', converted_pca.explained_variance_ratio_[0])

**Step 3 of 3: Compare the standardized values**

In [ ]:
# Compare the same observations after each independently fitted standardization.
standardized_difference = P_scaled - P_converted_scaled
absolute_standardized_difference = np.abs(standardized_difference)
largest_standardized_change = np.max(absolute_standardized_difference)
print('Largest standardized-input change:', largest_standardized_change)
scale_preview = pd.DataFrame(P[:4], columns=['Original feature 1', 'Original feature 2'])
scale_preview['Converted feature 1'] = P_converted[:4, 0]
display(scale_preview)

**Provided drawing: run unchanged.** This cell only displays the variables calculated above. Focus your experiment on the setting and model steps in the preceding cell.

In [ ]:
# Draw the already computed arrays; this cell contains no learning or scoring.
fig, axes = plt.subplots(1, 2, figsize=(10, 4))
plot_left = P_converted
plot_right = P_converted_scaled
plot_left_colors = np.zeros(240)
plot_right_colors = np.zeros(240)
# [:, 0] selects feature 1; [:, 1] selects feature 2 for every plotted row.
axes[0].scatter(plot_left[:, 0], plot_left[:, 1], c=plot_left_colors, cmap='tab10', s=15, alpha=0.75)
axes[1].scatter(plot_right[:, 0], plot_right[:, 1], c=plot_right_colors, cmap='tab10', s=15, alpha=0.75)
axes[0].set_title('Converted raw coordinates')
axes[1].set_title('Standardized coordinates')
axes[0].set_xlabel('Feature 1 (dimensionless)')
axes[1].set_xlabel('Feature 1 (dimensionless)')
axes[0].set_ylabel('Feature 2 (dimensionless)')
axes[1].set_ylabel('Feature 2 (dimensionless)')
# Equal coordinate scales preserve distance geometry within each panel.
axes[0].set_aspect('equal', adjustable='datalim')
axes[1].set_aspect('equal', adjustable='datalim')
fig.tight_layout()
# PROVIDED: red segments show two standard deviations along PC1 in each coordinate system.
raw_direction = converted_pca.components_[0] * np.sqrt(converted_pca.explained_variance_[0]) * 2
scaled_direction = scaled_pca.components_[0] * np.sqrt(scaled_pca.explained_variance_[0]) * 2
axes[0].plot(converted_pca.mean_[0] + [-raw_direction[0], raw_direction[0]],
             converted_pca.mean_[1] + [-raw_direction[1], raw_direction[1]], 'r-')
axes[1].plot([-scaled_direction[0], scaled_direction[0]], [-scaled_direction[1], scaled_direction[1]], 'r-')
plt.show()

**Personal note:** TODO: Record the main difference and its explanation in one or two sentences. Check the solution after attempting the experiment.

<a id="a2"></a>
## A2: How many components?

PCA reduces the number of feature coordinates. Explained variance reports
how much training variation each direction accounts for. Cumulative variance helps
describe compression; it does not establish classification or detection quality.
For this descriptive example, all of R is a learning illustration, not a held-out evaluation.

**Completed example:** Run and inspect the output.

In [ ]:
# Keep all three directions first so their variance fractions can be inspected.
# R_scaled remains a (240, 3) array; these are standardized features, not PCA scores.

# Construct a scaler; fit below will learn a separate mean and scale for each feature.
r_scaler = StandardScaler()

# Learn a mean and standard deviation for each column of R; no row is transformed yet.
r_scaler.fit(R)

# R_scaled keeps the same rows and columns; subtract the fitted mean and divide by the fitted scale.
R_scaled = r_scaler.transform(R)

# Construct PCA first; n_components controls compression and full fixes the solver.
r_full = PCA(svd_solver='full')

# Learn principal directions and their variances from R_scaled; no reference labels are used.
r_full.fit(R_scaled)

# Assemble a small result table one named column at a time.
variance_table = pd.DataFrame()
variance_table['Component'] = [1, 2, 3]
variance_table['Variance fraction'] = r_full.explained_variance_ratio_
variance_table['Cumulative fraction'] = np.cumsum(r_full.explained_variance_ratio_)
display(variance_table)

**Provided drawing: run unchanged.** This cell only displays the variables calculated above. Focus your experiment on the setting and model steps in the preceding cell.

In [ ]:
plt.bar([1, 2, 3], r_full.explained_variance_ratio_)
plt.plot([1, 2, 3], variance_table['Cumulative fraction'], 'ro-')
plt.xticks([1, 2, 3])
plt.xlabel('Component')
plt.ylabel('Variance fraction')
plt.title('Individual variance and cumulative variance')
plt.show()

**Your experiment:** Change component_count from 1 to 2. Compare score shape and retained variance.

In [ ]:
# Only the retained component count changes; the input rows and scaler stay fixed.
# A score column is a learned coordinate and is not one selected original feature.

# TODO: compare 1 with 2.
component_count = 1

# Construct PCA first; n_components controls compression and full fixes the solver.
r_reduced = PCA(n_components=component_count, svd_solver='full')

# Learn principal directions and their variances from R_scaled; no reference labels are used.
r_reduced.fit(R_scaled)

# R_scores has one score column per retained PC; apply the fitted directions without relearning them.
R_scores = r_reduced.transform(R_scaled)
print('Input shape:', R_scaled.shape)
print('Score shape:', R_scores.shape)
retained_variance_fraction = np.sum(r_reduced.explained_variance_ratio_)
print('Retained variance:', retained_variance_fraction)

**Personal note:** TODO: Record the main difference and its explanation in one or two sentences. Check the solution after attempting the experiment.

<a id="a3"></a>
## A3: Projection and reconstruction

Transform gives score coordinates; inverse_transform maps them back into
the original feature dimension. Here that original dimension is the **standardized**
three-feature space. Reconstruction error averages squared feature residuals
separately for each observation (`axis=1`), so its units are dimensionless.

**Completed example:** Run and inspect the output.

A2 examined the number and variance of score columns. Here the new question is how much feature information is lost when those scores are reconstructed.

In [ ]:
# Reconstruct three standardized features from a single PCA coordinate.
# Residuals compare values in the standardized feature space, not score space.

# Construct PCA first; n_components controls compression and full fixes the solver.
r_one = PCA(n_components=1, svd_solver='full')

# Learn principal directions and their variances from R_scaled; no reference labels are used.
r_one.fit(R_scaled)

# R_one_scores has one score column per retained PC; apply the fitted directions without relearning them.
R_one_scores = r_one.transform(R_scaled)

# R_one_reconstruction has the original feature-column count; reconstruct in the same feature space used for fitting.
R_one_reconstruction = r_one.inverse_transform(R_one_scores)
# Compare original and reconstructed values in the same feature space.
R_one_errors_difference = R_scaled - R_one_reconstruction
R_one_errors_squared = R_one_errors_difference ** 2
# axis=1 averages across feature columns, leaving one score per observation.
R_one_errors = np.mean(R_one_errors_squared, axis=1)
R_one_mean_error = np.mean(R_one_errors)
print('One-PC mean squared reconstruction error:', R_one_mean_error)

**Your experiment:** Change reconstruction_components to 2. Keep the same standardized inputs. Compare errors and original/reconstructed feature 3.

In [ ]:
# Use the same standardized data to isolate the effect of component count.
# Reconstruction still has three feature columns even when scores have fewer columns.

# TODO: compare 1 with 2.
reconstruction_components = 1

# Construct PCA first; n_components controls compression and full fixes the solver.
r_compare = PCA(n_components=reconstruction_components, svd_solver='full')

# Learn principal directions and their variances from R_scaled; no reference labels are used.
r_compare.fit(R_scaled)

# R_compare_scores has one score column per retained PC; apply the fitted directions without relearning them.
R_compare_scores = r_compare.transform(R_scaled)

# R_compare_reconstruction has the original feature-column count; reconstruct in the same feature space used for fitting.
R_compare_reconstruction = r_compare.inverse_transform(R_compare_scores)
# Compare original and reconstructed values in the same feature space.
R_compare_errors_difference = R_scaled - R_compare_reconstruction
R_compare_errors_squared = R_compare_errors_difference ** 2
# axis=1 averages across feature columns, leaving one score per observation.
R_compare_errors = np.mean(R_compare_errors_squared, axis=1)
baseline_mean_error = np.mean(R_one_errors)
print('Baseline MSE:', baseline_mean_error)
comparison_mean_error = np.mean(R_compare_errors)
print('Comparison MSE:', comparison_mean_error)

**Provided drawing: run unchanged.** This cell only displays the variables calculated above. Focus your experiment on the setting and model steps in the preceding cell.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(9, 4))
axes[0].scatter(R_scaled[:, 2], R_one_reconstruction[:, 2], s=12)
axes[1].scatter(R_scaled[:, 2], R_compare_reconstruction[:, 2], s=12)
axes[0].set_title('One-PC reconstruction')
axes[1].set_title('Comparison reconstruction')
axes[0].set_xlabel('Original standardized feature 3')
axes[1].set_xlabel('Original standardized feature 3')
axes[0].set_ylabel('Reconstructed feature 3')
axes[1].set_ylabel('Reconstructed feature 3')
axes[0].plot([-4, 4], [-4, 4], 'k--')
axes[1].plot([-4, 4], [-4, 4], 'k--')
fig.tight_layout()
plt.show()

**Personal note:** TODO: Record the main difference and its explanation in one or two sentences. Check the solution after attempting the experiment.

<a id="b1"></a>
## B1: Nearest centroids and one update

K-means alternates nearest-centroid assignment and updating each centroid
to its assigned points' mean. The supplied statements calculate feature-1 and
feature-2 differences separately for centroid 0, 1 and 2. Squaring and adding gives
one squared distance per observation and centroid. The distance table has one row
per observation and three centroid columns. `argmin` with `axis=1` chooses the
nearest centroid within each row. Run these supplied distance calculations unchanged.

**Completed example:** Run and inspect the output.

In [ ]:
# Each centroid has two coordinates, matching the two columns of K.
# The nearest centroid is assigned using squared distance in this original feature space.

# PROVIDED distance calculation: no model fitting is hidden here.
centers_before = np.array([[-3, 0], [3, -2], [3, 2]], dtype=float)
# Each distance array has 240 values: one squared distance per observation.
# Use the two feature columns explicitly instead of a three-dimensional broadcast.
# Compare all observations with centroid 0; each difference uses one feature column.
before_difference_1_0 = K[:, 0] - centers_before[0, 0]
before_difference_2_0 = K[:, 1] - centers_before[0, 1]
before_distance_0 = before_difference_1_0 ** 2 + before_difference_2_0 ** 2
# Compare all observations with centroid 1; each difference uses one feature column.
before_difference_1_1 = K[:, 0] - centers_before[1, 0]
before_difference_2_1 = K[:, 1] - centers_before[1, 1]
before_distance_1 = before_difference_1_1 ** 2 + before_difference_2_1 ** 2
# Compare all observations with centroid 2; each difference uses one feature column.
before_difference_1_2 = K[:, 0] - centers_before[2, 0]
before_difference_2_2 = K[:, 1] - centers_before[2, 1]
before_distance_2 = before_difference_1_2 ** 2 + before_difference_2_2 ** 2
# The three columns now mean squared distance to centroid 0, 1 and 2.
distances_before = np.column_stack([before_distance_0, before_distance_1, before_distance_2])

# axis=1 selects the smallest centroid distance within each observation row.
labels_before = np.argmin(distances_before, axis=1)
original_assignment_counts = np.bincount(labels_before)
print('Counts assigned to the original centroids:', original_assignment_counts)

**Step 2 of 2: Inspect a complete K-means fit**

In [ ]:
# A complete library fit repeats assignment/update until its stopping rule.

# Construct K-means with explicit k, initialization policy, restart count and seed.
k_baseline = KMeans(n_clusters=3, init='k-means++', n_init=10, random_state=42)

# Learn centroid locations from K coordinates only; reference labels do not enter fitting.
k_baseline.fit(K)

# Assign one cluster ID per observation using the fitted centroids.
k_baseline_labels = k_baseline.predict(K)

**Your experiment:** Move centroid 2 from [3, 2] to [-1, 1]. Inspect changed assignments and the following single centroid update. This is not a complete convergence run.

In [ ]:
# Move one initial centroid while keeping all observations fixed.
# We show one assignment and one mean update, not a full convergence run.

centers_moved = centers_before.copy()
# TODO: replace [3, 2] with [-1, 1].
centers_moved[2] = [3, 2]
# PROVIDED: distance and mean-update infrastructure, valid for the prescribed centroids.
# Each distance array has 240 values: one squared distance per observation.
# Use the two feature columns explicitly instead of a three-dimensional broadcast.
# Compare all observations with centroid 0; each difference uses one feature column.
moved_difference_1_0 = K[:, 0] - centers_moved[0, 0]
moved_difference_2_0 = K[:, 1] - centers_moved[0, 1]
moved_distance_0 = moved_difference_1_0 ** 2 + moved_difference_2_0 ** 2
# Compare all observations with centroid 1; each difference uses one feature column.
moved_difference_1_1 = K[:, 0] - centers_moved[1, 0]
moved_difference_2_1 = K[:, 1] - centers_moved[1, 1]
moved_distance_1 = moved_difference_1_1 ** 2 + moved_difference_2_1 ** 2
# Compare all observations with centroid 2; each difference uses one feature column.
moved_difference_1_2 = K[:, 0] - centers_moved[2, 0]
moved_difference_2_2 = K[:, 1] - centers_moved[2, 1]
moved_distance_2 = moved_difference_1_2 ** 2 + moved_difference_2_2 ** 2
# The three columns now mean squared distance to centroid 0, 1 and 2.
distances_moved = np.column_stack([moved_distance_0, moved_distance_1, moved_distance_2])

# axis=1 selects the smallest centroid distance within each observation row.
labels_moved = np.argmin(distances_moved, axis=1)

**Step 2 of 2: Update each centroid to its assigned mean**

In [ ]:
# A Boolean mask selects the rows assigned to one centroid.
group_0_mask = labels_moved == 0
group_1_mask = labels_moved == 1
group_2_mask = labels_moved == 2
group_0_points = K[group_0_mask]
group_1_points = K[group_1_mask]
group_2_points = K[group_2_mask]
# axis=0 averages down rows, giving a mean for each of the two feature columns.
new_center_0 = np.mean(group_0_points, axis=0)
new_center_1 = np.mean(group_1_points, axis=0)
new_center_2 = np.mean(group_2_points, axis=0)
centers_updated = np.vstack([new_center_0, new_center_1, new_center_2])
# True means that this observation changed its assigned centroid.
changed_assignment_mask = labels_before != labels_moved
changed_assignment_count = np.sum(changed_assignment_mask)
print('Changed assignments:', changed_assignment_count)
print('Updated centroids:', centers_updated)

**Provided drawing: run unchanged.** This cell only displays the variables calculated above. Focus your experiment on the setting and model steps in the preceding cell.

In [ ]:
# Draw the already computed arrays; this cell contains no learning or scoring.
fig, axes = plt.subplots(1, 2, figsize=(10, 4))
plot_left = K
plot_right = K
plot_left_colors = labels_moved
plot_right_colors = labels_moved
# [:, 0] selects feature 1; [:, 1] selects feature 2 for every plotted row.
axes[0].scatter(plot_left[:, 0], plot_left[:, 1], c=plot_left_colors, cmap='tab10', s=15, alpha=0.75)
axes[1].scatter(plot_right[:, 0], plot_right[:, 1], c=plot_right_colors, cmap='tab10', s=15, alpha=0.75)
axes[0].set_title('Moved centroids: assignment')
axes[1].set_title('One mean update: same assignments')
axes[0].set_xlabel('Feature 1 (dimensionless)')
axes[1].set_xlabel('Feature 1 (dimensionless)')
axes[0].set_ylabel('Feature 2 (dimensionless)')
axes[1].set_ylabel('Feature 2 (dimensionless)')
# Equal coordinate scales preserve distance geometry within each panel.
axes[0].set_aspect('equal', adjustable='datalim')
axes[1].set_aspect('equal', adjustable='datalim')
fig.tight_layout()
axes[0].scatter(centers_moved[:, 0], centers_moved[:, 1], marker='X', c='black', s=100)
axes[1].scatter(centers_updated[:, 0], centers_updated[:, 1], marker='X', c='black', s=100)
plt.show()

**Personal note:** TODO: Record the main difference and its explanation in one or two sentences. Check the solution after attempting the experiment.

<a id="b2"></a>
## B2: Choosing k

Inertia is the total squared distance to assigned centroids. More centroids
can reduce the optimal inertia simply by splitting groups. Silhouette compares
within-group closeness with separation from other groups. It requires at least
two populated clusters and fewer clusters than observations. Neither metric
proves the number of physical operating states.

**Completed example:** Run and inspect the output.

In [ ]:
# The same original K coordinates are used for both fits.
# Comparing inertia is meaningful here because the feature space and observations agree.

# Construct K-means with explicit k, initialization policy, restart count and seed.
k_two = KMeans(n_clusters=2, init='k-means++', n_init=10, random_state=42)

# Learn centroid locations from K coordinates only; reference labels do not enter fitting.
k_two.fit(K)

# Assign one cluster ID per observation using the fitted centroids.
k_two_labels = k_two.predict(K)

# Summarize within-cluster closeness versus separation in the same fitted feature space.
silhouette_two = silhouette_score(K, k_two_labels)

silhouette_three = silhouette_score(K, k_baseline_labels)
print('k=2 inertia:', k_two.inertia_, 'silhouette:', silhouette_two)
print('k=3 inertia:', k_baseline.inertia_, 'silhouette:', silhouette_three)

**Your experiment:** Set comparison_k to 4 and compare with k=2 and k=3. Explain why lower inertia alone is not a sufficient selection rule.

In [ ]:
# Change k without changing data, initialization policy or random seed.
# More clusters can reduce inertia by dividing a group; inspect silhouette and the plot too.

# TODO: change 3 to 4.
comparison_k = 3

# Construct K-means with explicit k, initialization policy, restart count and seed.
k_compare = KMeans(n_clusters=comparison_k, init='k-means++', n_init=10, random_state=42)

# Learn centroid locations from K coordinates only; reference labels do not enter fitting.
k_compare.fit(K)

# Assign one cluster ID per observation using the fitted centroids.
k_compare_labels = k_compare.predict(K)

# Summarize within-cluster closeness versus separation in the same fitted feature space.
silhouette_compare = silhouette_score(K, k_compare_labels)

# Assemble a small result table one named column at a time.
k_table = pd.DataFrame()
k_table['k'] = [2, 3, comparison_k]
k_table['Inertia'] = [k_two.inertia_, k_baseline.inertia_, k_compare.inertia_]
k_table['Silhouette'] = [silhouette_two, silhouette_three, silhouette_compare]
display(k_table)

**Provided drawing: run unchanged.** This cell only displays the variables calculated above. Focus your experiment on the setting and model steps in the preceding cell.

In [ ]:
# PROVIDED drawing: show all three choices once, using the same original coordinates.
fig, axes = plt.subplots(1, 3, figsize=(12, 4))
axes[0].scatter(K[:, 0], K[:, 1], c=k_two_labels, cmap='tab10', s=12)
axes[1].scatter(K[:, 0], K[:, 1], c=k_baseline_labels, cmap='tab10', s=12)
axes[2].scatter(K[:, 0], K[:, 1], c=k_compare_labels, cmap='tab10', s=12)
axes[0].set_title('k=2: fewer clusters')
axes[1].set_title('k=3: baseline')
axes[2].set_title('Comparison k')
axes[0].set_xlabel('Feature 1 (dimensionless)')
axes[1].set_xlabel('Feature 1 (dimensionless)')
axes[2].set_xlabel('Feature 1 (dimensionless)')
axes[0].set_ylabel('Feature 2 (dimensionless)')
axes[1].set_ylabel('Feature 2 (dimensionless)')
axes[2].set_ylabel('Feature 2 (dimensionless)')
axes[0].set_aspect('equal', adjustable='datalim')
axes[1].set_aspect('equal', adjustable='datalim')
axes[2].set_aspect('equal', adjustable='datalim')
fig.tight_layout()
plt.show()

**Personal note:** TODO: Record the main difference and its explanation in one or two sentences. Check the solution after attempting the experiment.

<a id="b3"></a>
## B3: Scaling changes distance

Euclidean distance uses numerical feature differences. A large feature scale
can dominate K-means even if that scale is arbitrary. We fit in the declared input
space, then display all partitions in the same original coordinates. PCA is not
a required step before K-means.

**Completed example:** Run and inspect the output.

A1 examined scaling for PCA variance. Here the new question is how scaling changes K-means distance and membership.

In [ ]:
# Standardize the feature columns before learning centroids.
# This fit uses two standardized features, not two PCA scores.

# Construct a scaler; fit below will learn a separate mean and scale for each feature.
k_scaler = StandardScaler()

# Learn a mean and standard deviation for each column of K; no row is transformed yet.
k_scaler.fit(K)

# K_scaled keeps the same rows and columns; subtract the fitted mean and divide by the fitted scale.
K_scaled = k_scaler.transform(K)

# Construct K-means with explicit k, initialization policy, restart count and seed.
k_standard = KMeans(n_clusters=3, init='k-means++', n_init=10, random_state=42)

# Learn centroid locations from K_scaled coordinates only; reference labels do not enter fitting.
k_standard.fit(K_scaled)

# Assign one cluster ID per observation using the fitted centroids.
k_standard_labels = k_standard.predict(K_scaled)
# Convert centers back to the original coordinate system for drawing only.

# Undo feature scaling for drawing centroids in the original K coordinates only.
k_centers_display = k_scaler.inverse_transform(k_standard.cluster_centers_)

**Your experiment:** Set distance_scale to 20. Compare the unstandardized inflated-feature fit with the standardized fit. Do not compare inertia values across the two metrics.

In [ ]:
# Inflate only feature 1 and inspect how the Euclidean distance changes.
# Draw both fits in original coordinates so different axis units do not obscure the comparison.

# TODO: change 1 to 20.
distance_scale = 1
# Change only feature 1, while preserving K for the shared display axes.
K_inflated = K.copy()
K_inflated[:, 0] = K[:, 0] * distance_scale

# Construct K-means with explicit k, initialization policy, restart count and seed.
k_inflated = KMeans(n_clusters=3, init='k-means++', n_init=10, random_state=42)

# Learn centroid locations from K_inflated coordinates only; reference labels do not enter fitting.
k_inflated.fit(K_inflated)

# Assign one cluster ID per observation using the fitted centroids.
k_inflated_labels = k_inflated.predict(K_inflated)

**Provided drawing: run unchanged.** This cell only displays the variables calculated above. Focus your experiment on the setting and model steps in the preceding cell.

In [ ]:
# Draw the already computed arrays; this cell contains no learning or scoring.
fig, axes = plt.subplots(1, 2, figsize=(10, 4))
plot_left = K
plot_right = K
plot_left_colors = k_inflated_labels
plot_right_colors = k_standard_labels
# [:, 0] selects feature 1; [:, 1] selects feature 2 for every plotted row.
axes[0].scatter(plot_left[:, 0], plot_left[:, 1], c=plot_left_colors, cmap='tab10', s=15, alpha=0.75)
axes[1].scatter(plot_right[:, 0], plot_right[:, 1], c=plot_right_colors, cmap='tab10', s=15, alpha=0.75)
axes[0].set_title('Inflated-feature fit: original axes')
axes[1].set_title('Standardized fit: original axes')
axes[0].set_xlabel('Feature 1 (dimensionless)')
axes[1].set_xlabel('Feature 1 (dimensionless)')
axes[0].set_ylabel('Feature 2 (dimensionless)')
axes[1].set_ylabel('Feature 2 (dimensionless)')
# Equal coordinate scales preserve distance geometry within each panel.
axes[0].set_aspect('equal', adjustable='datalim')
axes[1].set_aspect('equal', adjustable='datalim')
fig.tight_layout()
axes[1].scatter(k_centers_display[:, 0], k_centers_display[:, 1], marker='X', c='black', s=100)
plt.show()

**Personal note:** TODO: Record the main difference and its explanation in one or two sentences. Check the solution after attempting the experiment.

<a id="b4"></a>
## B4: Clusters and reference classes

Cluster IDs are arbitrary names, not known classes. Reference labels can be
revealed after fitting to inspect correspondence. A cross-tab shows the number
of observations in each cluster/class combination; raw cluster-ID accuracy would
mistake a harmless renaming for an error.

**Completed example:** Run and inspect the output.

In [ ]:
# Rename cluster 0 as 2, cluster 1 as 0, and cluster 2 as 1.
cluster_renaming = np.array([2, 0, 1])
# Array indexing applies the same lookup to every cluster ID; membership is unchanged.
permuted_labels = cluster_renaming[k_baseline_labels]
print('First eight cluster names:', k_baseline_labels[:8])
print('Same partition with renamed clusters:', permuted_labels[:8])

**Your experiment:** Change group_spread from 0.45 to 1.8. Fit on the overlapping points without labels, then inspect the class/cluster cross-tab.

In [ ]:
# Increasing group_spread makes the same three generating classes overlap.
# Fit K-means using coordinates only; reference labels are used afterward for interpretation.

# TODO: change 0.45 to 1.8.
group_spread = 0.45
K_overlap, overlap_reference = make_blobs(n_samples=240,
    centers=[[-3, 0], [3, -2], [3, 2]], cluster_std=group_spread, random_state=803)

# Construct K-means with explicit k, initialization policy, restart count and seed.
overlap_model = KMeans(n_clusters=3, init='k-means++', n_init=10, random_state=42)

# Learn centroid locations from K_overlap coordinates only; reference labels do not enter fitting.
overlap_model.fit(K_overlap)

# Assign one cluster ID per observation using the fitted centroids.
overlap_labels = overlap_model.predict(K_overlap)
overlap_table = pd.crosstab(overlap_labels, overlap_reference,
                            rownames=['Cluster'], colnames=['Reference class'])
display(overlap_table)

**Provided drawing: run unchanged.** This cell only displays the variables calculated above. Focus your experiment on the setting and model steps in the preceding cell.

In [ ]:
# Draw the already computed arrays; this cell contains no learning or scoring.
fig, axes = plt.subplots(1, 2, figsize=(10, 4))
plot_left = K_overlap
plot_right = K_overlap
plot_left_colors = overlap_reference
plot_right_colors = overlap_labels
# [:, 0] selects feature 1; [:, 1] selects feature 2 for every plotted row.
axes[0].scatter(plot_left[:, 0], plot_left[:, 1], c=plot_left_colors, cmap='tab10', s=15, alpha=0.75)
axes[1].scatter(plot_right[:, 0], plot_right[:, 1], c=plot_right_colors, cmap='tab10', s=15, alpha=0.75)
axes[0].set_title('Reference classes: display only')
axes[1].set_title('Clusters fit without labels')
axes[0].set_xlabel('Feature 1 (dimensionless)')
axes[1].set_xlabel('Feature 1 (dimensionless)')
axes[0].set_ylabel('Feature 2 (dimensionless)')
axes[1].set_ylabel('Feature 2 (dimensionless)')
# Equal coordinate scales preserve distance geometry within each panel.
axes[0].set_aspect('equal', adjustable='datalim')
axes[1].set_aspect('equal', adjustable='datalim')
fig.tight_layout()
plt.show()

**Personal note:** TODO: Record the main difference and its explanation in one or two sentences. Check the solution after attempting the experiment.

### Provided normal-pattern data: run unchanged

Training has 200 normal observations. Validation and final demonstration test
each have 80 normal, 40 off-axis anomaly and 40 along-axis anomaly observations.
All are generated now, before fitting. Their seeds are independent.
Type 0 means normal, type 1 means a perpendicular departure, and type 2 means
an unusually large coordinate along the normal line. Both types 1 and 2 have
evaluation label 1 (anomaly). These are synthetic definitions, not real faults.

The supplied statements generate each group explicitly, without custom
functions. `vstack` joins observation rows; `repeat` supplies the evaluation
labels. No anomaly enters normal training. Run this setup unchanged.

In [ ]:
# PROVIDED: generate independent datasets before any scaler or PCA is fit.
# Every row is a synthetic observation; both feature columns are dimensionless.
training_rng = np.random.default_rng(810)
# Normal variation is broad along a line and narrow across that line.
train_along = training_rng.normal(0, 1, 200)
train_across = training_rng.normal(0, 0.10, 200)
train_feature_1 = (train_along + train_across) / np.sqrt(2)
train_feature_2 = (train_along - train_across) / np.sqrt(2)
N_train = np.column_stack([train_feature_1, train_feature_2])

**Provided validation data: run unchanged.** These independently generated normal and anomaly observations calibrate the threshold; they are not model-fitting rows.

In [ ]:
# Independent seed 811: no observation is reused from training.
validation_rng = np.random.default_rng(811)
# Generate 80 normal observations for validation.
validation_normal_along = validation_rng.normal(0, 1, 80)
validation_normal_across = validation_rng.normal(0, 0.10, 80)
validation_normal_feature_1 = (validation_normal_along + validation_normal_across) / np.sqrt(2)
validation_normal_feature_2 = (validation_normal_along - validation_normal_across) / np.sqrt(2)
validation_normal = np.column_stack([validation_normal_feature_1, validation_normal_feature_2])
# Generate 40 off axis observations for validation.
validation_off_axis_along = validation_rng.normal(0, 1, 40)
validation_off_axis_across = validation_rng.normal(0, 0.10, 40)
validation_off_axis_feature_1 = (validation_off_axis_along + validation_off_axis_across) / np.sqrt(2)
validation_off_axis_feature_2 = (validation_off_axis_along - validation_off_axis_across) / np.sqrt(2)
validation_off_axis = np.column_stack([validation_off_axis_feature_1, validation_off_axis_feature_2])
# Shift the two feature columns explicitly; do not rescale their noise.
validation_off_axis[:, 0] = validation_off_axis[:, 0] + 0.65 / np.sqrt(2)
validation_off_axis[:, 1] = validation_off_axis[:, 1] - 0.65 / np.sqrt(2)

**Step 2 of 2: Generate along-axis observations and attach labels**

In [ ]:
# Generate 40 along axis observations for validation.
validation_along_axis_along = validation_rng.normal(0, 1, 40)
validation_along_axis_across = validation_rng.normal(0, 0.10, 40)
validation_along_axis_feature_1 = (validation_along_axis_along + validation_along_axis_across) / np.sqrt(2)
validation_along_axis_feature_2 = (validation_along_axis_along - validation_along_axis_across) / np.sqrt(2)
validation_along_axis = np.column_stack([validation_along_axis_feature_1, validation_along_axis_feature_2])
validation_along_axis[:, 0] = validation_along_axis[:, 0] + 4.5 / np.sqrt(2)
validation_along_axis[:, 1] = validation_along_axis[:, 1] + 4.5 / np.sqrt(2)
# Stack the groups in a fixed order: normal, off-axis, then along-axis.
N_validation = np.vstack([validation_normal, validation_off_axis, validation_along_axis])
# Evaluation labels distinguish normal=0 from anomaly=1.
validation_labels = np.repeat([0, 1, 1], [80, 40, 40])
# Type codes distinguish the two synthetic anomaly mechanisms for interpretation.

# Type IDs describe synthetic mechanisms for plots; both anomaly types share evaluation label 1.

validation_types = np.repeat([0, 1, 2], [80, 40, 40])

**Provided test data: run unchanged.** Generate the observations now, but defer transformation, scoring and evaluation until C3.

In [ ]:
# Independent seed 812: no observation is reused from training.
test_rng = np.random.default_rng(812)
# Generate 80 normal observations for test.
test_normal_along = test_rng.normal(0, 1, 80)
test_normal_across = test_rng.normal(0, 0.10, 80)
test_normal_feature_1 = (test_normal_along + test_normal_across) / np.sqrt(2)
test_normal_feature_2 = (test_normal_along - test_normal_across) / np.sqrt(2)
test_normal = np.column_stack([test_normal_feature_1, test_normal_feature_2])
# Generate 40 off axis observations for test.
test_off_axis_along = test_rng.normal(0, 1, 40)
test_off_axis_across = test_rng.normal(0, 0.10, 40)
test_off_axis_feature_1 = (test_off_axis_along + test_off_axis_across) / np.sqrt(2)
test_off_axis_feature_2 = (test_off_axis_along - test_off_axis_across) / np.sqrt(2)
test_off_axis = np.column_stack([test_off_axis_feature_1, test_off_axis_feature_2])
# Shift the two feature columns explicitly; do not rescale their noise.
test_off_axis[:, 0] = test_off_axis[:, 0] + 0.65 / np.sqrt(2)
test_off_axis[:, 1] = test_off_axis[:, 1] - 0.65 / np.sqrt(2)

**Step 2 of 2: Generate along-axis observations and attach labels**

In [ ]:
# Generate 40 along axis observations for test.
test_along_axis_along = test_rng.normal(0, 1, 40)
test_along_axis_across = test_rng.normal(0, 0.10, 40)
test_along_axis_feature_1 = (test_along_axis_along + test_along_axis_across) / np.sqrt(2)
test_along_axis_feature_2 = (test_along_axis_along - test_along_axis_across) / np.sqrt(2)
test_along_axis = np.column_stack([test_along_axis_feature_1, test_along_axis_feature_2])
test_along_axis[:, 0] = test_along_axis[:, 0] + 4.5 / np.sqrt(2)
test_along_axis[:, 1] = test_along_axis[:, 1] + 4.5 / np.sqrt(2)
# Stack the groups in a fixed order: normal, off-axis, then along-axis.
N_test = np.vstack([test_normal, test_off_axis, test_along_axis])
# Evaluation labels distinguish normal=0 from anomaly=1.
test_labels = np.repeat([0, 1, 1], [80, 40, 40])
# Type codes distinguish the two synthetic anomaly mechanisms for interpretation.

# Type IDs describe synthetic mechanisms for plots; both anomaly types share evaluation label 1.

test_types = np.repeat([0, 1, 2], [80, 40, 40])
print('Normal training:', N_train.shape, 'Validation:', N_validation.shape, 'Test:', N_test.shape)

<a id="c1"></a>
## C1: Learn a normal pattern

Fit the scaler and PCA on **normal training only**. Validation is transformed
without refitting. A high residual means distance from the learned principal
subspace, not a fault probability. An unusual observation along that subspace
can still reconstruct well. All score calculations below use standardized features.

**Completed example:** Run and inspect the output.

In [ ]:
# N_train has 200 normal rows and two dimensionless feature columns.
# Validation anomalies must not influence the scaler means or PCA directions.

# Construct a scaler; fit below will learn a separate mean and scale for each feature.
normal_scaler = StandardScaler()

# Learn from normal training only; validation and test must not affect this step.
normal_scaler.fit(N_train)

# N_train_scaled keeps the same rows and columns; subtract the fitted mean and divide by the fitted scale.
N_train_scaled = normal_scaler.transform(N_train)

# N_validation_scaled keeps the same rows and columns; subtract the fitted mean and divide by the fitted scale.
N_validation_scaled = normal_scaler.transform(N_validation)

**Step 2 of 3: Fit PCA and calculate reconstruction residuals**

In [ ]:
# Construct PCA first; n_components controls compression and full fixes the solver.
normal_pca = PCA(n_components=1, whiten=False, svd_solver='full')

normal_pca.fit(N_train_scaled)

# train_scores has one score column per retained PC; apply the fitted directions without relearning them.
train_scores = normal_pca.transform(N_train_scaled)

# train_reconstructed has the original feature-column count; reconstruct in the same feature space used for fitting.
train_reconstructed = normal_pca.inverse_transform(train_scores)
# Compare original and reconstructed values in the same feature space.
train_residual_difference = N_train_scaled - train_reconstructed
train_residual_squared = train_residual_difference ** 2
# axis=1 averages across feature columns, leaving one score per observation.
train_residual = np.mean(train_residual_squared, axis=1)

# validation_scores has one score column per retained PC; apply the fitted directions without relearning them.
validation_scores = normal_pca.transform(N_validation_scaled)

# validation_reconstructed has the original feature-column count; reconstruct in the same feature space used for fitting.
validation_reconstructed = normal_pca.inverse_transform(validation_scores)
# axis=1 averages squared feature errors separately for each observation.
validation_residual_difference = N_validation_scaled - validation_reconstructed
validation_residual_squared = validation_residual_difference ** 2
validation_residual = np.mean(validation_residual_squared, axis=1)
print('Normal-training PC1 retained variance:', normal_pca.explained_variance_ratio_[0])

**Step 3 of 3: Summarize normal and off-axis residuals**

In [ ]:
# Select normal validation rows for interpretation, not for fitting.
validation_normal_mask = validation_types == 0
validation_normal_residual = validation_residual[validation_normal_mask]
validation_normal_median = np.median(validation_normal_residual)
print('Validation median residual, normal:', validation_normal_median)
# Select off-axis validation rows for interpretation, not for fitting.
validation_off_axis_mask = validation_types == 1
validation_off_axis_residual = validation_residual[validation_off_axis_mask]
validation_off_axis_median = np.median(validation_off_axis_residual)
print('Validation median residual, off-axis:', validation_off_axis_median)

**Your experiment:** Change review_type from 1 (off-axis anomaly) to 2 (along-axis anomaly). Compare each with the normal group and inspect the reconstruction segment. The plot marks one observation and its reconstruction.

In [ ]:
# review_type selects one anomaly mechanism for explanation, not for refitting.
# Masks contain True/False per observation; they select rows rather than feature columns.

# TODO: compare type 1 with type 2.
review_type = 1
review_mask = validation_types == review_type
normal_mask = validation_types == 0
# Boolean indexing keeps only the selected observations' residuals.
normal_review_residual = validation_residual[normal_mask]
anomaly_review_residual = validation_residual[review_mask]
normal_review_median = np.median(normal_review_residual)
anomaly_review_median = np.median(anomaly_review_residual)
print('Normal median residual:', normal_review_median)
print('Selected anomaly median residual:', anomaly_review_median)

**Provided drawing: run unchanged.** This cell only displays the variables calculated above. Focus your experiment on the setting and model steps in the preceding cell.

In [ ]:
# PROVIDED drawing: keep the full context but mark just one reconstruction.
plot_normal = N_validation_scaled[validation_types == 0]
plot_off_axis = N_validation_scaled[validation_types == 1]
plot_along_axis = N_validation_scaled[validation_types == 2]
fig, axis = plt.subplots(figsize=(7, 4))
axis.scatter(plot_normal[:, 0], plot_normal[:, 1], s=12, label='Normal')
axis.scatter(plot_off_axis[:, 0], plot_off_axis[:, 1], s=12, label='Off-axis anomaly')
axis.scatter(plot_along_axis[:, 0], plot_along_axis[:, 1], s=12, label='Along-axis anomaly')
# Choose the first row of the selected type for a single visible projection segment.
segment_rows = np.flatnonzero(review_mask)
segment_row = segment_rows[0]
segment_original = N_validation_scaled[segment_row]
segment_reconstruction = validation_reconstructed[segment_row]
axis.plot([segment_original[0], segment_reconstruction[0]],
          [segment_original[1], segment_reconstruction[1]], 'k-', linewidth=2)
axis.scatter(segment_reconstruction[0], segment_reconstruction[1], marker='x', c='black', s=55)
axis.set_xlabel('Standardized feature 1')
axis.set_ylabel('Standardized feature 2')
axis.set_title('One selected observation and its reconstruction')
axis.set_aspect('equal', adjustable='datalim')
axis.legend()
fig.tight_layout()
plt.show()

**Personal note:** TODO: Record the main difference and its explanation in one or two sentences. Check the solution after attempting the experiment.

<a id="c2"></a>
## C2: Select a threshold using validation

An observation is predicted anomalous when its residual is **greater than** the
threshold. We form candidates from training residual percentiles, then use labeled
validation data to calibrate the detector. Normal-only model fitting does not make
this entire workflow label-free. Training percentiles do not guarantee future
false alarm rates. Balanced accuracy is the average of normal recall and anomaly
recall. Select the largest validation balanced accuracy; tie-break by fewer false
positives, then the larger threshold. Confusion matrices use rows for actual labels,
columns for predicted labels, and label order 0=normal, 1=anomaly. Thus `[0, 1]`
counts false positives and `[1, 0]` counts missed anomalies.
The candidate table is provided infrastructure.

**Completed example:** Run and inspect the output.

In [ ]:
# Training percentiles supply three candidate residual thresholds.
# Validation labels help select a threshold; they do not enter scaler or PCA fitting.

# Use training scores to form this candidate; it does not guarantee future false alarm rates.
threshold_90 = np.percentile(train_residual, 90)

threshold_95 = np.percentile(train_residual, 95)

threshold_99 = np.percentile(train_residual, 99)

# A score greater than this candidate threshold is predicted anomalous (True).
validation_prediction_90 = validation_residual > threshold_90

validation_prediction_95 = validation_residual > threshold_95

validation_prediction_99 = validation_residual > threshold_99

**Step 2 of 3: Compare validation errors**

In [ ]:
# Compare validation predictions with labels, giving normal and anomaly recall equal weight.
balanced_90 = balanced_accuracy_score(validation_labels, validation_prediction_90)

balanced_95 = balanced_accuracy_score(validation_labels, validation_prediction_95)

balanced_99 = balanced_accuracy_score(validation_labels, validation_prediction_99)

# Rows are actual, columns are predicted, with normal=0 and anomaly=1 in this order.
matrix_90 = confusion_matrix(validation_labels, validation_prediction_90, labels=[0, 1])

matrix_95 = confusion_matrix(validation_labels, validation_prediction_95, labels=[0, 1])

matrix_99 = confusion_matrix(validation_labels, validation_prediction_99, labels=[0, 1])

**Step 3 of 3: Read the candidate table**

In [ ]:
# Assemble a small result table one named column at a time.
threshold_table = pd.DataFrame()
threshold_table['Training percentile'] = [90, 95, 99]
threshold_table['Threshold'] = [threshold_90, threshold_95, threshold_99]
threshold_table['Validation balanced accuracy'] = [balanced_90, balanced_95, balanced_99]
threshold_table['False positives'] = [matrix_90[0, 1], matrix_95[0, 1], matrix_99[0, 1]]
threshold_table['Missed anomalies'] = [matrix_90[1, 0], matrix_95[1, 0], matrix_99[1, 0]]
display(threshold_table)

**Your experiment:** Choose threshold_90, threshold_95 or threshold_99 using the stated validation rule. Change selected_threshold accordingly and explain the FP/FN trade-off before running C3.

In [ ]:
# Choose using the validation table before opening the final test section.
# The same threshold applies to every observation, regardless of its true type.

# TODO: replace the initial threshold using validation evidence.
selected_threshold = threshold_90
print('Selected threshold:', selected_threshold)

**Provided drawing: run unchanged.** This cell only displays the variables calculated above. Focus your experiment on the setting and model steps in the preceding cell.

In [ ]:
plt.figure(figsize=(8, 4))
# PROVIDED drawing only: the same bins support a fair distribution comparison.
shared_bins = np.linspace(0, np.max(validation_residual) * 1.01, 25)
plt.hist(validation_residual[validation_types == 0], bins=shared_bins, alpha=0.5, label='Normal')
plt.hist(validation_residual[validation_types == 1], bins=shared_bins, alpha=0.5, label='Off-axis anomaly')
plt.hist(validation_residual[validation_types == 2], bins=shared_bins, alpha=0.5, label='Along-axis anomaly')
plt.axvline(selected_threshold, color='black', linestyle='--', label='Selected threshold')
plt.xlabel('Mean squared standardized residual')
plt.ylabel('Observation count')
plt.title('Validation score distributions')
plt.legend()
plt.tight_layout()
plt.show()

**Personal note:** TODO: Record the main difference and its explanation in one or two sentences. Check the solution after attempting the experiment.

<a id="c3"></a>
## C3: Freeze, then evaluate

Run C2 and record your choice first. The next cell freezes it, transforms the
test using the training-fitted objects, and evaluates without refitting. Do not
reselect a threshold to improve these test results. Repeated Run all reproduces a
teaching example; it does not create a newly unseen test. Precision depends on the
synthetic anomaly proportion, which is not field prevalence.

**Completed example:** Run and inspect the output.

In [ ]:
# Copy the validation choice before calculating any final test score.
# Reuse the training-fitted scaler and PCA; never call fit on the test observations.

frozen_threshold = selected_threshold

# N_test_scaled keeps the same rows and columns; subtract the fitted mean and divide by the fitted scale.
N_test_scaled = normal_scaler.transform(N_test)

# test_scores has one score column per retained PC; apply the fitted directions without relearning them.
test_scores = normal_pca.transform(N_test_scaled)

# test_reconstructed has the original feature-column count; reconstruct in the same feature space used for fitting.
test_reconstructed = normal_pca.inverse_transform(test_scores)
# Compare original and reconstructed values in the same feature space.
test_residual_difference = N_test_scaled - test_reconstructed
test_residual_squared = test_residual_difference ** 2
# axis=1 averages across feature columns, leaving one score per observation.
test_residual = np.mean(test_residual_squared, axis=1)

# Apply the frozen rule once; True means predicted anomaly, not a probability.
test_prediction = test_residual > frozen_threshold

**Step 2 of 2: Report false alarms and missed anomalies**

In [ ]:
# Rows are actual, columns are predicted, with normal=0 and anomaly=1 in this order.
test_matrix = confusion_matrix(test_labels, test_prediction, labels=[0, 1])

# Precision asks what fraction of predicted anomalies are labeled anomalies; use zero for an empty denominator.
test_precision = precision_score(test_labels, test_prediction, zero_division=0)

# Recall asks what fraction of labeled anomalies the frozen detector finds.
test_recall = recall_score(test_labels, test_prediction, zero_division=0)
# Row 0 contains actual normal observations: correctly normal plus false alarms.
test_normal_count = np.sum(test_matrix[0])
test_false_positive_count = test_matrix[0, 1]
test_false_alarm_rate = test_false_positive_count / test_normal_count
print('Frozen threshold:', frozen_threshold)
print('Confusion matrix rows=actual, columns=predicted; order=normal, anomaly:')
print(test_matrix)
print('Normal false alarm rate:', test_false_alarm_rate)
print('Missed anomaly count:', test_matrix[1, 0])
print('Anomaly precision:', test_precision, 'Anomaly recall:', test_recall)
print('Precision is reported as zero if no anomalies are predicted.')

**Provided drawing: run unchanged.** This cell only displays the variables calculated above. Focus your experiment on the setting and model steps in the preceding cell.

In [ ]:
fig, axis = plt.subplots(figsize=(6, 4))
axis.scatter(np.arange(160), test_residual, c=test_types, cmap='tab10', s=15)
axis.axhline(frozen_threshold, color='black', linestyle='--')
axis.set_xlabel('Observation index: normal, off-axis, along-axis')
axis.set_ylabel('Mean squared standardized residual')
axis.set_title('Final demonstration test: frozen detector')
plt.tight_layout()
plt.show()

**Your experiment:** Compare type 1 and type 2 detection counts without changing the model or threshold. Set test_review_type to 2 after first inspecting type 1.

In [ ]:
# Inspect the kinds of missed anomalies without changing the frozen detector.
# Counting a selected type is interpretation, not another round of model selection.

# TODO: compare 1 with 2; keep the detector frozen.
test_review_type = 1
test_review_mask = test_types == test_review_type
print('Selected anomaly type:', test_review_type)
# Count predictions only for the requested anomaly type; do not change the detector.
selected_type_predictions = test_prediction[test_review_mask]
selected_type_detected = np.sum(selected_type_predictions)
selected_type_count = np.sum(test_review_mask)
print('Detected:', selected_type_detected, 'of', selected_type_count)

**Personal note:** TODO: Record the main difference and its explanation in one or two sentences. Check the solution after attempting the experiment.

<a id="basic-stop"></a>
## Basic route complete

You may stop here. The additional experiments are optional.

### Connection to Lab 7

| Here | Lab 7 |
|---|---|
| One synthetic observation | One MIMII recording |
| Supplied dimensionless coordinates | Documented acoustic feature values |
| Normal-only scaler/PCA fit | Same training restriction |
| Validation threshold, then frozen test | Same evaluation roles |
| Known synthetic generation | Real mechanisms and file-level labels require cautious interpretation |

Lab 7 concerns fan recordings within one machine ID; it will not establish
unseen-machine performance. No Lab partition IDs, file-specific answers or
selected Lab thresholds are provided here. PCA reconstruction is an anomaly
score, while K-means assigns groups. Neither automatically diagnoses faults.
A two-PC plot also does not prove that a clustering model used only two PCs.

<a id="additional-practice"></a>
## Additional experiments

Run the basic route first. D4 uses training and validation only; it does not
reopen the final test to choose between detectors.

<a id="d1"></a>
## D1: Variance is not class information

This is an intentional **unstandardized** counterexample. A broad nuisance
feature dominates variance; a narrow second feature separates reference classes.
The labels are never fit targets. This is not a claim about standardized PCA.

**Completed example:** Run and inspect the output.

In [ ]:
# This optional example deliberately keeps the original, unequal numerical scales.
# The narrow second feature carries class information, but PCA does not see class labels.

rng_d1 = np.random.default_rng(820)
d1_reference = np.repeat([0, 1], 120)
# Preserve the fixed RNG order: broad feature first, then narrow-feature noise.
d1_nuisance = rng_d1.normal(0, 8, 240)
d1_class_noise = rng_d1.normal(0, 0.12, 240)
# Class 0 is centered at -0.6; class 1 is centered at +0.6.
d1_class_centers = (d1_reference * 2 - 1) * 0.6
d1_narrow_feature = d1_class_centers + d1_class_noise
D1 = np.column_stack([d1_nuisance, d1_narrow_feature])

# Construct PCA first; n_components controls compression and full fixes the solver.
d1_pca = PCA(n_components=1, svd_solver='full')

# Learn principal directions and their variances from D1; no reference labels are used.
d1_pca.fit(D1)

# d1_scores has one score column per retained PC; apply the fitted directions without relearning them.
d1_scores = d1_pca.transform(D1)

# d1_reconstructed has the original feature-column count; reconstruct in the same feature space used for fitting.
d1_reconstructed = d1_pca.inverse_transform(d1_scores)
print('Raw PC1 retained variance:', d1_pca.explained_variance_ratio_[0])

**Provided drawing: run unchanged.** This cell only displays the variables calculated above. Focus your experiment on the setting and model steps in the preceding cell.

In [ ]:
# Draw the already computed arrays; this cell contains no learning or scoring.
fig, axes = plt.subplots(1, 2, figsize=(10, 4))
plot_left = D1
plot_right = d1_reconstructed
plot_left_colors = d1_reference
plot_right_colors = d1_reference
# [:, 0] selects feature 1; [:, 1] selects feature 2 for every plotted row.
axes[0].scatter(plot_left[:, 0], plot_left[:, 1], c=plot_left_colors, cmap='tab10', s=15, alpha=0.75)
axes[1].scatter(plot_right[:, 0], plot_right[:, 1], c=plot_right_colors, cmap='tab10', s=15, alpha=0.75)
axes[0].set_title('Original: reference labels revealed')
axes[1].set_title('One-PC reconstruction')
axes[0].set_xlabel('Feature 1 (dimensionless)')
axes[1].set_xlabel('Feature 1 (dimensionless)')
axes[0].set_ylabel('Feature 2 (dimensionless)')
axes[1].set_ylabel('Feature 2 (dimensionless)')
# Auto axis scales make the narrow class-separating coordinate visible in this raw-scale counterexample.
fig.tight_layout()
plt.show()

**Your experiment:** Change d1_components to 2 and compare reconstruction of the narrow class-separating direction. More retained variance is not a downstream-task guarantee.

In [ ]:
# Compare reconstruction with a different number of components.
# Revealed reference classes help explain which information is retained or discarded.

# TODO: change 1 to 2.
d1_components = 1

# Construct PCA first; n_components controls compression and full fixes the solver.
d1_compare = PCA(n_components=d1_components, svd_solver='full')

# Learn principal directions and their variances from D1; no reference labels are used.
d1_compare.fit(D1)

# d1_compare_scores has one score column per retained PC; apply the fitted directions without relearning them.
d1_compare_scores = d1_compare.transform(D1)

# d1_compare_reconstructed has the original feature-column count; reconstruct in the same feature space used for fitting.
d1_compare_reconstructed = d1_compare.inverse_transform(d1_compare_scores)

**Step 2 of 2: Compare the class-separating coordinate**

In [ ]:
# Reference masks are for interpretation after PCA fitting; they are not fit targets.
d1_class_0_mask = d1_reference == 0
d1_class_1_mask = d1_reference == 1
d1_original_0 = D1[d1_class_0_mask, 1]
d1_original_1 = D1[d1_class_1_mask, 1]
d1_reconstructed_0 = d1_compare_reconstructed[d1_class_0_mask, 1]
d1_reconstructed_1 = d1_compare_reconstructed[d1_class_1_mask, 1]
# Compare the class means in feature 2 before and after reconstruction.
d1_original_mean_0 = np.mean(d1_original_0)
d1_original_mean_1 = np.mean(d1_original_1)
d1_reconstructed_mean_0 = np.mean(d1_reconstructed_0)
d1_reconstructed_mean_1 = np.mean(d1_reconstructed_1)
d1_original_gap = d1_original_mean_1 - d1_original_mean_0
d1_reconstructed_gap = d1_reconstructed_mean_1 - d1_reconstructed_mean_0
print('Original class mean gap in feature 2:', d1_original_gap)
print('Reconstructed class mean gap in feature 2:', d1_reconstructed_gap)

**Personal note:** TODO: Record the main difference and its explanation in one or two sentences. Check the solution after attempting the experiment.

<a id="d2"></a>
## D2: Initialization and local solutions

K-means can stop at different local solutions. Here we deliberately use
random initialization and one start, then compare multiple starts. Seeds are fixed
for illustration; not every dataset or pair of seeds produces different results.

**Completed example:** Run and inspect the output.

In [ ]:
# Keep the synthetic observations fixed and use one random centroid initialization.
# Unequal group sizes make a poor local solution visible for selected seeds.

D2, d2_reference = make_blobs(n_samples=[180, 180, 40],
    centers=[[-6, 0], [0, 0], [6, 0]], cluster_std=0.6, random_state=821)

# Construct K-means with explicit k, initialization policy, restart count and seed.
d2_baseline = KMeans(n_clusters=3, init='random', n_init=1, random_state=0)

# Learn centroid locations from D2 coordinates only; reference labels do not enter fitting.
d2_baseline.fit(D2)

# Assign one cluster ID per observation using the fitted centroids.
d2_baseline_labels = d2_baseline.predict(D2)
print('Seed 0, one start inertia:', d2_baseline.inertia_)

**Your experiment:** Change initialization_seed to 3, compare one start with the baseline, then inspect ten starts using that same seed.

In [ ]:
# Change only the initialization seed, then the number of starts.
# n_init=10 keeps the lowest-inertia result among ten starts for the same data.

# TODO: change 0 to 3.
initialization_seed = 0

# Construct K-means with explicit k, initialization policy, restart count and seed.
d2_single = KMeans(n_clusters=3, init='random', n_init=1, random_state=initialization_seed)

# Learn centroid locations from D2 coordinates only; reference labels do not enter fitting.
d2_single.fit(D2)

# Assign one cluster ID per observation using the fitted centroids.
d2_single_labels = d2_single.predict(D2)

d2_multiple = KMeans(n_clusters=3, init='random', n_init=10, random_state=initialization_seed)

d2_multiple.fit(D2)

d2_multiple_labels = d2_multiple.predict(D2)
print('One start inertia:', d2_single.inertia_)
print('Ten starts inertia:', d2_multiple.inertia_)

**Provided drawing: run unchanged.** This cell only displays the variables calculated above. Focus your experiment on the setting and model steps in the preceding cell.

In [ ]:
# Draw the already computed arrays; this cell contains no learning or scoring.
fig, axes = plt.subplots(1, 2, figsize=(10, 4))
plot_left = D2
plot_right = D2
plot_left_colors = d2_single_labels
plot_right_colors = d2_multiple_labels
# [:, 0] selects feature 1; [:, 1] selects feature 2 for every plotted row.
axes[0].scatter(plot_left[:, 0], plot_left[:, 1], c=plot_left_colors, cmap='tab10', s=15, alpha=0.75)
axes[1].scatter(plot_right[:, 0], plot_right[:, 1], c=plot_right_colors, cmap='tab10', s=15, alpha=0.75)
axes[0].set_title('One random start')
axes[1].set_title('Best of ten random starts')
axes[0].set_xlabel('Feature 1 (dimensionless)')
axes[1].set_xlabel('Feature 1 (dimensionless)')
axes[0].set_ylabel('Feature 2 (dimensionless)')
axes[1].set_ylabel('Feature 2 (dimensionless)')
# Equal coordinate scales preserve distance geometry within each panel.
axes[0].set_aspect('equal', adjustable='datalim')
axes[1].set_aspect('equal', adjustable='datalim')
fig.tight_layout()
plt.show()

**Personal note:** TODO: Record the main difference and its explanation in one or two sentences. Check the solution after attempting the experiment.

<a id="d3"></a>
## D3: When K-means geometry fails

Two curved generating groups do not match the compact centroid geometry
favored by K-means. We fit without labels and reveal the generating groups for
interpretation. No additional clustering algorithm is required.

**Completed example:** Run and inspect the output.

In [ ]:
# These two curved groups are synthetic and have no physical fault interpretation.
# K-means uses their two feature columns without their generating group labels.

D3, d3_reference = make_moons(n_samples=300, noise=0.06, random_state=822)

# Construct K-means with explicit k, initialization policy, restart count and seed.
d3_model = KMeans(n_clusters=2, init='k-means++', n_init=10, random_state=42)

# Learn centroid locations from D3 coordinates only; reference labels do not enter fitting.
d3_model.fit(D3)

# Assign one cluster ID per observation using the fitted centroids.
d3_labels = d3_model.predict(D3)
d3_cluster_ids = np.unique(d3_labels)
print('Fitted clusters:', d3_cluster_ids)

**Your experiment:** Replace right_coloring with d3_labels to compare the reference grouping with the fitted partition. Explain the membership mismatch rather than the color names.

In [ ]:
# Only the plot coloring changes; the already fitted model stays fixed.
# Compare group membership rather than treating matching color names as matching classes.

# TODO: replace d3_reference with d3_labels.
right_coloring = d3_reference

**Provided drawing: run unchanged.** This cell only displays the variables calculated above. Focus your experiment on the setting and model steps in the preceding cell.

In [ ]:
# Draw the already computed arrays; this cell contains no learning or scoring.
fig, axes = plt.subplots(1, 2, figsize=(10, 4))
plot_left = D3
plot_right = D3
plot_left_colors = d3_reference
plot_right_colors = right_coloring
# [:, 0] selects feature 1; [:, 1] selects feature 2 for every plotted row.
axes[0].scatter(plot_left[:, 0], plot_left[:, 1], c=plot_left_colors, cmap='tab10', s=15, alpha=0.75)
axes[1].scatter(plot_right[:, 0], plot_right[:, 1], c=plot_right_colors, cmap='tab10', s=15, alpha=0.75)
axes[0].set_title('Generating groups')
axes[1].set_title('Comparison grouping')
axes[0].set_xlabel('Feature 1 (dimensionless)')
axes[1].set_xlabel('Feature 1 (dimensionless)')
axes[0].set_ylabel('Feature 2 (dimensionless)')
axes[1].set_ylabel('Feature 2 (dimensionless)')
# Equal coordinate scales preserve distance geometry within each panel.
axes[0].set_aspect('equal', adjustable='datalim')
axes[1].set_aspect('equal', adjustable='datalim')
fig.tight_layout()
plt.show()

**Personal note:** TODO: Record the main difference and its explanation in one or two sentences. Check the solution after attempting the experiment.

<a id="d4"></a>
## D4: More components and a second detector

Full-rank PCA can reconstruct both normal and anomalous inputs nearly
exactly, leaving almost no residual for detection. Isolation Forest instead uses
random partitioning. We use negative score_samples so larger scores mean more
anomalous, fit on the same normal training, and calibrate its own threshold.
This optional comparison uses validation only and cannot establish detector superiority.

**Completed example:** Run and inspect the output.

This is an optional comparison to read, not another required workflow to master.

In [ ]:
# Isolation Forest fits the same normal training rows as PCA.
# It uses all standardized features and has its own score units and validation thresholds.

d4_one_median = np.median(validation_residual)
print('One-PC validation median residual:', d4_one_median)
if_model = IsolationForest(n_estimators=100, max_samples='auto',
    contamination='auto', random_state=42)

# Learn from normal training only; validation and test must not affect this step.
if_model.fit(N_train_scaled)

# Negate the library score so larger values mean more anomalous, matching the PCA direction.
if_train_score = -if_model.score_samples(N_train_scaled)

if_validation_score = -if_model.score_samples(N_validation_scaled)

**Step 2 of 4: Form its threshold candidates**

In [ ]:
# Use training scores to form this candidate; it does not guarantee future false alarm rates.
if_threshold_90 = np.percentile(if_train_score, 90)

if_threshold_95 = np.percentile(if_train_score, 95)

if_threshold_99 = np.percentile(if_train_score, 99)

# A score greater than this candidate threshold is predicted anomalous (True).
if_prediction_90 = if_validation_score > if_threshold_90

if_prediction_95 = if_validation_score > if_threshold_95

if_prediction_99 = if_validation_score > if_threshold_99

**Step 3 of 4: Compare validation errors**

In [ ]:
# Compare validation predictions with labels, giving normal and anomaly recall equal weight.
if_balanced_90 = balanced_accuracy_score(validation_labels, if_prediction_90)

if_balanced_95 = balanced_accuracy_score(validation_labels, if_prediction_95)

if_balanced_99 = balanced_accuracy_score(validation_labels, if_prediction_99)

# Rows are actual, columns are predicted, with normal=0 and anomaly=1 in this order.
if_matrix_90 = confusion_matrix(validation_labels, if_prediction_90, labels=[0, 1])

if_matrix_95 = confusion_matrix(validation_labels, if_prediction_95, labels=[0, 1])

if_matrix_99 = confusion_matrix(validation_labels, if_prediction_99, labels=[0, 1])

**Step 4 of 4: Read its candidate table**

In [ ]:
# Assemble a small result table one named column at a time.
if_table = pd.DataFrame()
if_table['Percentile'] = [90, 95, 99]
if_table['Threshold'] = [if_threshold_90, if_threshold_95, if_threshold_99]
if_table['Validation balanced accuracy'] = [if_balanced_90, if_balanced_95, if_balanced_99]
if_table['False positives'] = [if_matrix_90[0, 1], if_matrix_95[0, 1], if_matrix_99[0, 1]]
if_table['Missed anomalies'] = [if_matrix_90[1, 0], if_matrix_95[1, 0], if_matrix_99[1, 0]]
display(if_table)

**Your experiment:** Set d4_components to 2 and inspect the residual. Separately choose the IF threshold using the same validation rule as C2. Do not evaluate IF on the final test.

In [ ]:
# Changing the component count tests whether residuals still carry information.
# This optional comparison uses validation only and must not reopen the final test.

# TODO: change 1 to 2.
d4_components = 1

# Construct PCA first; n_components controls compression and full fixes the solver.
d4_pca = PCA(n_components=d4_components, svd_solver='full')

# Learn from normal training only; validation and test must not affect this step.
d4_pca.fit(N_train_scaled)

# d4_scores has one score column per retained PC; apply the fitted directions without relearning them.
d4_scores = d4_pca.transform(N_validation_scaled)

# d4_reconstructed has the original feature-column count; reconstruct in the same feature space used for fitting.
d4_reconstructed = d4_pca.inverse_transform(d4_scores)
# Compare original and reconstructed values in the same feature space.
d4_residual_difference = N_validation_scaled - d4_reconstructed
d4_residual_squared = d4_residual_difference ** 2
# axis=1 averages across feature columns, leaving one score per observation.
d4_residual = np.mean(d4_residual_squared, axis=1)
d4_maximum_residual = np.max(d4_residual)
print('Comparison maximum residual:', d4_maximum_residual)

**Step 2 of 2: Inspect the selected IF threshold**

In [ ]:
# TODO: choose from IF validation evidence; keep its own score units.
if_selected_threshold = if_threshold_90
if_validation_prediction = if_validation_score > if_selected_threshold
# Select the normal observations and count their anomaly predictions.
if_normal_mask = validation_types == 0
if_normal_prediction = if_validation_prediction[if_normal_mask]
if_normal_count = np.sum(if_normal_prediction)
print('IF normal false alarms:', if_normal_count)
# Select the off axis observations and count their anomaly predictions.
if_off_axis_mask = validation_types == 1
if_off_axis_prediction = if_validation_prediction[if_off_axis_mask]
if_off_axis_count = np.sum(if_off_axis_prediction)
print('IF off-axis anomalies detected:', if_off_axis_count)
# Select the along axis observations and count their anomaly predictions.
if_along_axis_mask = validation_types == 2
if_along_axis_prediction = if_validation_prediction[if_along_axis_mask]
if_along_axis_count = np.sum(if_along_axis_prediction)
print('IF along-axis anomalies detected:', if_along_axis_count)

**Provided drawing: run unchanged.** This cell only displays the variables calculated above. Focus your experiment on the setting and model steps in the preceding cell.

In [ ]:
fig, axis = plt.subplots(figsize=(7, 4))
axis.scatter(validation_residual, if_validation_score, c=validation_types, cmap='tab10', s=15)
axis.axvline(selected_threshold, color='black', linestyle='--', label='PCA validation threshold')
axis.axhline(if_selected_threshold, color='gray', linestyle='--', label='IF validation threshold')
axis.set_xlabel('PCA standardized residual')
axis.set_ylabel('Isolation Forest: negative score_samples')
axis.set_title('Development observations: different scores')
axis.legend()
plt.tight_layout()
plt.show()

**Personal note:** TODO: Record the main difference and its explanation in one or two sentences. Check the solution after attempting the experiment.

## Personal self-check

Rendered notebook checkboxes are not clickable. To record completion, edit
this Markdown cell and change `[ ]` to `[x]`.

- [ ] I compared PCA direction, score shape and reconstruction.
- [ ] I explored centroids, k, scaling and cluster/class interpretation.
- [ ] I distinguished normal-only fitting from label-informed calibration.
- [ ] I selected the threshold before inspecting the final test.
- [ ] I can explain a PCA residual failure case and a K-means geometry limitation.

This Practice is optional and ungraded. No upload or separate report is required.

## References and data attribution

All numerical examples are course-created synthetic illustrations. They do not
use a redistributed manufacturing dataset. scikit-learn's make_blobs and
make_moons generate the named synthetic examples.

- [PCA and decomposition](https://scikit-learn.org/stable/modules/decomposition.html)
- [K-means and clustering](https://scikit-learn.org/stable/modules/clustering.html)
- [StandardScaler](https://scikit-learn.org/stable/modules/generated/sklearn.preprocessing.StandardScaler.html)
- [IsolationForest](https://scikit-learn.org/stable/modules/generated/sklearn.ensemble.IsolationForest.html)
- Lab context only: [MIMII source](https://zenodo.org/records/3384388)

Numeric results can differ slightly across package versions. PCA sign changes
and K-means cluster-name permutations do not necessarily change the learned structure.